BGE

In [1]:
import numpy as np 
from sentence_transformers import SentenceTransformer
from variable_share import corpus 
import torch 

c:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chemin utilisé : C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\chroma_3gpp_rel18
Existe : True

Contenu du dossier :
- ._chroma.sqlite3
- ._eae17378-2ffa-4d50-9800-01eaa05f2742
- chroma.sqlite3
- eae17378-2ffa-4d50-9800-01eaa05f2742

Nombre de collections : 1
[Collection(name=3gpp_rel18_collection)]
Collection(name=3gpp_rel18_collection)

Nombre de chunk : 562865
{'ids': ['2865084', '2865085', '2865086', '2865087', '2865088', '2865089', '2865090', '2865091', '2865092', '2865093'], 'embeddings': array([[-0.0092853 ,  0.01435731, -0.00187875, ...,  0.07671405,
         0.05635262, -0.00273231],
       [ 0.05003922,  0.0073945 , -0.01688451, ...,  0.00345017,
         0.04526629,  0.02224259],
       [-0.00206988,  0.02758421,  0.03543698, ..., -0.04465876,
         0.00797065,  0.01978247],
       ...,
       [-0.07843515,  0.01852478,  0.04259703, ...,  0.04828352,
         0.0727049 ,  0.02867837],
       [-0.09269179,  0.03357768,  0.0112438 , ...,  0.0197269

In [2]:
# 1. Vérifier si un GPU NVIDIA est disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Calculs exécutés sur : {device}")

Calculs exécutés sur : cuda


In [4]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5" , device ="cuda")
# Documents
bge_embeddings = model.encode(corpus , normalize_embeddings = True)
print(bge_embeddings.shape)

# Question
question = "What is a Change Request?"
query_embedding = model.encode(question , normalize_embeddings = True)
print(query_embedding.shape)



#Calculate embedding similarities
scores_bge  = model.similarity(query_embedding  , bge_embeddings )
print(scores_bge)



(5000, 384)
(384,)
tensor([[0.5941, 0.5836, 0.6620,  ..., 0.4825, 0.4419, 0.4739]])


In [8]:
scores_bge = model.similarity(query_embedding, bge_embeddings)[0].cpu().numpy()

top_indices_bge = np.argsort(scores_bge)[-5:][::-1]

for i in top_indices_bge:
    print("Score BGE :", scores_bge[i])
    print(corpus[i])
    print()

Score BGE : 0.7487358
Section: 4.6	Change Request regime > 4.6.2	Change Request forms
Table Analysis:
Caption: Table 4A: Categories of Change Requests
Data:
| Category | Meaning                    | Remarks                    |
+----------+----------------------------+----------------------------+
|          |                            | > leads to incorrect       |
|          |                            | > operation of the         |
|          |                            | > system); or              |
|          |                            | >                          |
|          |                            | > 2 to correct an          |
|          |                            | > ambiguity in the         |
|          |                            | > specification which      |
|          |                            | > could lead to different  |
|          |                            | > implementations which    |
|          |                            | > cannot inter-opera

BM25

In [9]:
%pip install rank-bm25
from rank_bm25 import BM25Okapi 
from chroma import ma_collection
import string
import re
import numpy as np 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from rank_bm25 import BM25Okapi
import re

# récupérer les documents
data = ma_collection.get(
    limit=5000,
    include=["documents"]
)

corpus = data["documents"]

# fonction pour découper le texte en mots
def tokenize(text):
    return re.findall(r"[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*", text.lower())

# tokeniser tous les documents
tokenized_corpus = [tokenize(doc) for doc in corpus]

# créer BM25
bm25 = BM25Okapi(tokenized_corpus)

Test

In [11]:
question = "What is a Change Request?"

question_tokenized = tokenize(question)

scores = bm25.get_scores(question_tokenized)

Top K

In [12]:
top_indices_bm25 = np.argsort(scores)[::-1][:5]

for i in top_indices_bm25:
    print("Score :", scores[i])
    print(corpus[i])
    print()

Score : 13.048107483030694
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - + | pcr | pseudo - change - request : similar to a change | | | request but has no cr number and is intended | | | to propose new or revised text for inclusion | | | in 3gpp tss or trs not yet under change |

Score : 12.696033919347556
Section: 4.6	Change Request regime > 4.6.2	Change Request forms
Table Analysis:
Caption: Table 4A: Categories of Change Requests
Data:
| Category | Meaning                    | Remarks                    |
+----------+----------------------------+----------------------------+
|          |                            | > leads to incorrect       |
|          |                            | > operation of the         |
|          |                            | > system); or              |
|          |                            | >                          |
|          |                            | > 2 to correct an          |
|          |                            | > a

Fusion with *Reciprocal Rank Fusion (RRF)* 

In [13]:
def rrf_fusion(bm25_indices, bge_indices, k=60, top_k=5):

    scores = {}

    # BM25
    for rank, idx in enumerate(bm25_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # BGE
    for rank, idx in enumerate(bge_indices, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    # Trier du meilleur score au plus faible
    ranking_final = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    
    return ranking_final[:top_k]    

In [14]:

resultats_rrf = rrf_fusion(
    top_indices_bm25,
    top_indices_bge,
    top_k=5
)

print(resultats_rrf)

[(np.int64(131), 0.03252247488101534), (np.int64(22), 0.031009615384615385), (np.int64(141), 0.01639344262295082), (np.int64(86), 0.016129032258064516), (np.int64(142), 0.015873015873015872)]


In [16]:
for idx, score in resultats_rrf:
    print("Score RRF :", score)
    print(corpus[idx])
    print()

Score RRF : 0.03252247488101534
Section: 4.6	Change Request regime > 4.6.2	Change Request forms
Table Analysis:
Caption: Table 4A: Categories of Change Requests
Data:
| Category | Meaning                    | Remarks                    |
+----------+----------------------------+----------------------------+
|          |                            | > leads to incorrect       |
|          |                            | > operation of the         |
|          |                            | > system); or              |
|          |                            | >                          |
|          |                            | > 2 to correct an          |
|          |                            | > ambiguity in the         |
|          |                            | > specification which      |
|          |                            | > could lead to different  |
|          |                            | > implementations which    |
|          |                            | > cannot i